<a href="https://colab.research.google.com/github/Pooja7101/AI-Review-Recommendation-System/blob/main/04_absa_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentence-transformers spacy

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

from transformers import pipeline

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis"
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
def extract_aspects(review):

    doc = nlp(review)

    aspects = []

    for chunk in doc.noun_chunks:

        aspects.append(chunk.text.lower())

    return list(set(aspects))

In [ ]:
review = """
The cleaner cleans well but the oiliness it leaves is high.
"""

print(extract_aspects(review))

['it', '\nthe cleaner cleans', 'the oiliness']


In [ ]:
positive_words = [
    'good',
    'great',
    'perfect',
    'excellent',
    'amazing',
    'comfortable',
    'fits perfect',
    'fit perfectly',
    'love'
]

negative_words = [
    'bad',
    'poor',
    'terrible',
    'delayed',
    'delay',
    'late',
    'cheap',
    'small',
    'broken',
    'hate'
]


def get_aspect_sentiment(review, aspect):

    review_lower = review.lower()

    aspect_clean = aspect.replace("the ", "").strip()

    words = review_lower.split()

    aspect_words = aspect_clean.split()

    aspect_index = -1

    # find aspect position

    for i in range(len(words)):

        window = " ".join(words[i:i+len(aspect_words)])

        if aspect_clean in window:

            aspect_index = i

            break

    # fallback

    if aspect_index == -1:

        relevant_text = review_lower

    else:

        # smaller local context window

        start = max(0, aspect_index - 2)

        end = min(len(words), aspect_index + 5)

        relevant_text = " ".join(words[start:end])

    print("Relevant Text:", relevant_text)

    # scoring

    positive_score = 0
    negative_score = 0

    # positive word scoring

    for word in positive_words:

        if word in relevant_text:
            positive_score += 1

    # negative word scoring

    for word in negative_words:

        if word in relevant_text:
            negative_score += 1

    # final decision

    if negative_score > positive_score:
        return "NEGATIVE"

    elif positive_score > negative_score:
        return "POSITIVE"

    # transformer fallback

    result = sentiment_pipeline(relevant_text)[0]

    return result['label']

In [ ]:
review = """
The battery life is fantastic and the display looks beautiful,
but the speakers sound terrible and the phone heats up quickly.
"""

In [ ]:
aspects = extract_aspects(review)

print(aspects)

['the phone', 'the speakers', 'the display', '\nthe battery life']


In [ ]:
for aspect in aspects:

    sentiment = get_aspect_sentiment(
        review,
        aspect
    )

    print(f"Aspect: {aspect}")

    print(f"Sentiment: {sentiment}")

    print("-"*40)

Relevant Text: and the phone heats up quickly.
Aspect: the phone
Sentiment: POSITIVE
----------------------------------------
Relevant Text: but the speakers sound terrible and the
Aspect: the speakers
Sentiment: NEGATIVE
----------------------------------------
Relevant Text: and the display looks beautiful, but the
Aspect: the display
Sentiment: POSITIVE
----------------------------------------
Relevant Text: the battery life is fantastic and
Aspect: 
the battery life
Sentiment: POSITIVE
----------------------------------------


In [ ]:
absa_results = []

for aspect in aspects:

    sentiment = get_aspect_sentiment(
        review,
        aspect
    )

    absa_results.append({
        'aspect': aspect,
        'sentiment': sentiment
    })

print(absa_results)

Relevant Text: and the phone heats up quickly.
Relevant Text: but the speakers sound terrible and the
Relevant Text: and the display looks beautiful, but the
Relevant Text: the battery life is fantastic and
[{'aspect': 'the phone', 'sentiment': 'POSITIVE'}, {'aspect': 'the speakers', 'sentiment': 'NEGATIVE'}, {'aspect': 'the display', 'sentiment': 'POSITIVE'}, {'aspect': '\nthe battery life', 'sentiment': 'POSITIVE'}]


In [ ]:
import pandas as pd

from transformers import pipeline

from google.colab import drive

drive.mount('/content/drive')

# LOAD DATASET

df = pd.read_csv('/content/drive/MyDrive/balanced_reviews_6k.csv')


# TRANSFORMER SENTIMENT PIPELINE

sentiment_pipeline = pipeline(
    "sentiment-analysis"
)


# DOMAIN ASPECTS

fashion_aspects = [

    'fit',
    'size',
    'comfort',
    'material',
    'quality',
    'design',
    'style',
    'color',
    'delivery',
    'price',
    'durability',
    'fabric',
    'shoe',
    'sole',
    'heel',
    'jewelry',
    'bracelet',
    'ring',
    'chain',
    'packaging'
]


# POSITIVE WORDS

positive_words = [

    'good',
    'great',
    'perfect',
    'excellent',
    'amazing',
    'comfortable',
    'premium',
    'beautiful',
    'love',
    'soft',
    'stylish',
    'durable',
    'fits perfect',
    'fit perfectly'
]


# NEGATIVE WORDS

negative_words = [

    'bad',
    'poor',
    'terrible',
    'delayed',
    'delay',
    'late',
    'cheap',
    'small',
    'broken',
    'hate',
    'tight',
    'uncomfortable',
    'damaged',
    'peeling',
    'loose',
    'disappointing'
]


# ASPECT EXTRACTION

def extract_aspects(review):

    review_lower = review.lower()

    found_aspects = []

    for aspect in fashion_aspects:

        if aspect in review_lower:

            found_aspects.append(aspect)

    return list(set(found_aspects))


# ASPECT SENTIMENT DETECTION

def get_aspect_sentiment(review, aspect):

    review_lower = review.lower()

    words = review_lower.split()

    aspect_words = aspect.split()

    aspect_index = -1


    # FIND ASPECT POSITION

    for i in range(len(words)):

        window = " ".join(words[i:i+len(aspect_words)])

        if aspect in window:

            aspect_index = i

            break


    # LOCAL CONTEXT WINDOW

    if aspect_index == -1:

        relevant_text = review_lower

    else:

        start = max(0, aspect_index - 2)

        end = min(len(words), aspect_index + 5)

        relevant_text = " ".join(words[start:end])


    # SENTIMENT SCORING

    positive_score = 0

    negative_score = 0


    # POSITIVE WORD MATCHING

    for word in positive_words:

        if word in relevant_text:

            positive_score += 1


    # NEGATIVE WORD MATCHING

    for word in negative_words:

        if word in relevant_text:

            negative_score += 1


    # FINAL DECISION

    if negative_score > positive_score:

        return "NEGATIVE"


    elif positive_score > negative_score:

        return "POSITIVE"


    # TRANSFORMER FALLBACK

    result = sentiment_pipeline(relevant_text)[0]

    return result['label']


# SMALL SAMPLE FIRST

df_small = df.head(100).copy()


# PROCESS REVIEWS

all_results = []


for index, row in df_small.iterrows():

    review = row['cleaned_review']

    aspects = extract_aspects(review)


    for aspect in aspects:

        sentiment = get_aspect_sentiment(

            review,

            aspect
        )


        all_results.append({

            'review': review,

            'aspect': aspect,

            'aspect_sentiment': sentiment
        })


# CREATE DATAFRAME

absa_df = pd.DataFrame(all_results)


# VIEW RESULTS

print(absa_df.head(20))


# SAVE RESULTS

absa_df.to_csv(

    '/content/drive/MyDrive/absa_results.csv',

    index=False
)

Mounted at /content/drive


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

                                               review    aspect  \
0   instead of going to a jewelry store to purchas...  bracelet   
1   instead of going to a jewelry store to purchas...   jewelry   
2   sadly had to return as toe area too shallow we...      shoe   
3   sadly had to return as toe area too shallow we...       fit   
4   sadly had to return as toe area too shallow we...      ring   
5   my husband and i both desperately needed slipp...     color   
6   my husband and i both desperately needed slipp...     price   
7   my husband and i both desperately needed slipp...      size   
8   my husband and i both desperately needed slipp...   comfort   
9   my husband and i both desperately needed slipp...      shoe   
10  well this dress was only about and you can tel...  material   
11  to my surprise it fit like it was made for me ...       fit   
12  i purchased this ring and i was shocked how so...      ring   
13  i purchased this ring and i was shocked how so...   jewelr